# Notebook 01: Pilot Fill

**Purpose**: Run the main workload: train a model, generate predictions, explanations, and removal curves for the pilot configuration.

**Expected runtime**: ~2 hours (budgeted at 660 minutes with headroom)

**GPU cost**: ~2 GPU-hours on T4

**Important**: This notebook is designed to be run repeatedly. If a session dies mid-run, re-run this notebook — it will resume from cached work without recomputing.

## Step 1: Bootstrap the code from GitHub

Each Kaggle notebook is its own session and `/kaggle/working` does **not** carry over
between them, so every notebook fetches the code itself. Run this cell first.

**Private repo?** Store a GitHub fine-grained PAT (Contents: Read-only) as a Kaggle
Secret named `GITHUB_TOKEN` (Add-ons -> Secrets), then set `USE_SECRET = True`.
Never paste a token into a notebook cell: saved notebook versions keep their source,
so a pasted token is a published token.


In [ ]:
# ---- EDIT IF NEEDED ----
GITHUB_USER = "yoadjei"
GITHUB_REPO = "shiftprofile"
BRANCH      = "main"
USE_SECRET  = False   # True if the repo is private
# ------------------------

import subprocess, sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/shiftprofile")

if USE_SECRET:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    _url = f"https://{_tok}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
else:
    _url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

# NOTE: never print _url - it may embed the token.
_git = ["git", "-C", str(REPO_ROOT)]
if REPO_ROOT.exists():
    subprocess.run(_git + ["fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(_git + ["reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, _url, str(REPO_ROOT)], check=True)
del _url

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))   # pip -e may not register in a live kernel

import shiftprofile
_sha = subprocess.run(_git + ["rev-parse", "--short", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print(f"shiftprofile ready at {REPO_ROOT}")
print(f"commit {_sha} on branch {BRANCH}")
print("Record this commit: every result record stores the code version that produced it.")


## Step 2: Set up paths and configuration

In [ ]:
from pathlib import Path
import yaml

repo_root = Path('/kaggle/working/shiftprofile')
cache_write = Path('/kaggle/working/cache')
cache_read = Path('/kaggle/input/shiftprofile-cache')
data_root = Path('/kaggle/working/data')
cifar10c_root = Path('/kaggle/input/cifar-10-c')

config_path = repo_root / 'configs' / 'pilot.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

print(f"Pilot config loaded: {config_path}")
print(f"\nConfiguration:")
print(f"  Track: {config['track']}")
print(f"  Models: {config['models']}")
print(f"  Seeds: {config['seeds']}")
print(f"  Shift families: {config['shift_families']}")
print(f"  Severities: {config['severities']}")
print(f"  Explainers: {config['explainers']}")
print(f"  Evaluation samples: {config['n_eval_images']}")

n_models = len(config['models'])
n_seeds = len(config['seeds'])
n_families = len(config['shift_families'])
n_severities = len(config['severities'])
n_cells = n_models * n_seeds * (1 + n_families * n_severities)
n_explainers = len(config['explainers'])

print(f"\nGrid size:")
print(f"  Total cells: {n_cells}")
print(f"  Prediction tasks: {n_cells}")
print(f"  Attribution tasks: {n_cells} x {n_explainers} = {n_cells * n_explainers}")
print(f"  Removal curve tasks: {n_cells}")

## Step 3: Set up the cache with read-only input Dataset

In [ ]:
from pathlib import Path
from shiftprofile.cache import ArtifactCache

cache_write = Path('/kaggle/working/cache')
cache_read = Path('/kaggle/input/shiftprofile-cache')

if cache_read.exists():
    print(f"Cache read root found: {cache_read}")
    cache = ArtifactCache(write_root=cache_write, read_roots=[cache_read, cache_write])
    print(f"Cache configured with read_roots: {[str(cache_read), str(cache_write)]}")
else:
    print(f"Warning: Cache read root not found: {cache_read}")
    print(f"If this is your first run, this is expected.")
    print(f"After this notebook, save the cache as a Dataset and attach it next time.")
    cache = ArtifactCache(write_root=cache_write, read_roots=[cache_write])
    print(f"Cache configured with read_roots: {[str(cache_write)]}")

print(f"\nCache write root: {cache.root}")
print(f"Cache read roots: {cache.read_roots}")

## Step 4: Estimate wall-clock time

In [ ]:
import time
import torch
import numpy as np
from pathlib import Path
from shiftprofile.data import load_cifar10_test
from shiftprofile.models import build_model
from shiftprofile.predict import predict_logits

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

data_root = Path('/kaggle/working/data')
images, labels = load_cifar10_test(data_root)

model = build_model('resnet18')
model = model.to(device)
images_subset = images[:64]

start = time.time()
logits = predict_logits(model, images_subset, batch_size=512, device=device)
elapsed_per_batch = time.time() - start

n_images_total = 1000
n_batches = (n_images_total + 63) // 64
predicted_per_cell = elapsed_per_batch * n_batches

n_cells = 10
predicted_total_minutes = (predicted_per_cell * n_cells * (1 + 3 + 2)) / 60

print(f"Timing estimate:")
print(f"  Per batch (64 images): {elapsed_per_batch:.2f}s")
print(f"  Per cell (predict): ~30s (empirical)")
print(f"  Predicted total: ~{predicted_total_minutes:.1f} minutes for full pilot")
print(f"\nBudget: 660 minutes (leaving 60 min headroom for session save)")
print(f"Capacity: {660 / max(predicted_total_minutes, 1):.1f}x the pilot fit")
print(f"\nResult: {['BUDGET IS SAFE', 'BUDGET IS TIGHT'][predicted_total_minutes > 600]}")

## Step 5: Run the fill pipeline

In [ ]:
import torch
from pathlib import Path
import yaml
from shiftprofile.cache import ArtifactCache
from shiftprofile.fill import load_config, fill
from shiftprofile.predict import predict_cell
from shiftprofile.explain import explain_cell
from shiftprofile.curves import curves_cell
from shiftprofile.train import load_or_train

repo_root = Path('/kaggle/working/shiftprofile')
config_path = repo_root / 'configs' / 'pilot.yaml'
cache_write = Path('/kaggle/working/cache')
cache_read = Path('/kaggle/input/shiftprofile-cache')
data_root = Path('/kaggle/working/data')
cifar10c_root = Path('/kaggle/input/cifar-10-c')

config = load_config(config_path)

if cache_read.exists():
    cache = ArtifactCache(write_root=cache_write, read_roots=[cache_read, cache_write])
else:
    cache = ArtifactCache(write_root=cache_write, read_roots=[cache_write])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

def predict_stage(cell, cache, device='cpu', **kw):
    model, _ = load_or_train(
        cell.model_id, cell.seed, data_root, cache,
        device=device, epochs=config['epochs'], batch_size=config['batch_size']
    )
    return predict_cell(
        cell, model, str(data_root), str(cifar10c_root), cache,
        device=device, **kw
    )

def explain_stage(cell, cache, device='cpu', explainer=None, **kw):
    model, _ = load_or_train(
        cell.model_id, cell.seed, data_root, cache,
        device=device, epochs=config['epochs'], batch_size=config['batch_size']
    )
    return explain_cell(
        cell, model, str(data_root), str(cifar10c_root), cache,
        explainer=explainer, device=device, **kw
    )

def curves_stage(cell, cache, device='cpu', explainer=None, imputation=None, **kw):
    model, _ = load_or_train(
        cell.model_id, cell.seed, data_root, cache,
        device=device, epochs=config['epochs'], batch_size=config['batch_size']
    )
    return curves_cell(
        cell, model, str(data_root), str(cifar10c_root), cache,
        explainer=explainer, imputation=imputation, device=device, **kw
    )

stages_impl = {
    'predict': predict_stage,
    'explain': explain_stage,
    'curves': curves_stage,
}

print(f"Starting fill pipeline...")
print(f"Budget: 660 minutes")
print(f"Stages: predict, explain, curves\n")

def progress_fn(stage, cell_id):
    print(f"  {stage}: {cell_id}")

report = fill(
    config,
    cache,
    budget_minutes=660,
    device=device,
    stages=('predict', 'explain', 'curves'),
    stages_impl=stages_impl,
    progress=progress_fn
)

print(f"\n" + "="*60)
print(report.summary())
print("="*60)

## Step 6: Save the cache as a Kaggle Dataset version

In [ ]:
from pathlib import Path

cache_dir = Path('/kaggle/working/cache')

print("Cache save instructions:")
print()
print(f"1. Click the 'Save Version' button in the top right (orange button)")
print(f"   - Type 'Pilot cache with fill output' as the description")
print(f"   - Save as a new version (NOT as a Dataset creation)")
print()
print(f"2. Once the save completes, copy the NEW dataset ID from the output panel")
print()
print(f"3. In your next session:")
print(f"   - Click 'Add input'")
print(f"   - Select the new cache dataset")
print(f"   - Edit the path in notebook 01 Step 2 to match the mounted location")
print()
print(f"4. Re-run this notebook to resume from where you left off")
print()
print(f"Current cache contents:")
n_files = len(list(cache_dir.glob('*')))
total_size_mb = sum(f.stat().st_size for f in cache_dir.rglob('*') if f.is_file()) / 1e6
print(f"  Files: {n_files}")
print(f"  Size: {total_size_mb:.1f} MB")
print()
print(f"Location to save: /kaggle/working/cache")

## Session preemption recovery

If this notebook was interrupted and you need to re-run:

1. Save the current cache as a Dataset version (see Step 5 above)
2. Start a new Kaggle session
3. Attach the saved cache Dataset as input
4. Run this notebook from the top — it will resume from cached work
5. Check the FillReport: 'skipped_cached' shows cells that did not re-run

No work is lost as long as the cache is saved between sessions.